# 🫀 Heart Disease Prediction — Random Forest Training
### Google Colab Notebook | Dataset: 1024 Pasien

**Alur Training:**
1. Install & Import Libraries
2. Load Dataset
3. Data Cleaning & Preprocessing
4. Exploratory Data Analysis (EDA)
5. Feature Encoding & Scaling
6. Train-Test Split
7. Training Random Forest
8. Hyperparameter Tuning (GridSearchCV)
9. Evaluasi Model (Accuracy, Precision, Recall, F1, ROC-AUC)
10. Visualisasi Hasil (Confusion Matrix, ROC Curve, Feature Importance)
11. Simpan Model `.pkl` untuk Deployment Django

## 1️⃣ Install & Import Libraries

In [2]:
# Install library tambahan jika belum ada di Colab
!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib

# ── Core Libraries
import os
import warnings
import numpy as np
import pandas as pd

# ── Visualisasi
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── Scikit-learn: Preprocessing
from sklearn.model_selection import (
    train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline

# ── Scikit-learn: Model
from sklearn.ensemble import RandomForestClassifier

# ── Scikit-learn: Evaluasi
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    confusion_matrix, classification_report
)

# ── Save model
import joblib
from google.colab import files

# ── Setting
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Palette warna premium
COLOR_POS  = '#E63946'   # Sakit jantung
COLOR_NEG  = '#2A9D8F'   # Tidak sakit
COLOR_MAIN = '#1A3A5C'   # Biru tua
COLOR_ACC  = '#F4A261'   # Orange aksen
PALETTE    = [COLOR_NEG, COLOR_POS]

print('✅ Semua library berhasil diimport!')


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'google'

## 2️⃣ Load Dataset
> Upload file `heart.csv` dari lokal, atau mount Google Drive.

In [ ]:
# ================================================================
# PILIH SALAH SATU METODE LOAD DATASET:
# ================================================================

# ── OPSI A: Upload langsung dari komputer
uploaded = files.upload()  # Akan muncul tombol pilih file
filename = list(uploaded.keys())[0]
df_raw = pd.read_csv(filename)

# ── OPSI B: Mount Google Drive (uncomment jika pakai Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# df_raw = pd.read_csv('/content/drive/MyDrive/heart.csv')

# ── OPSI C: Pakai URL langsung
# df_raw = pd.read_csv('https://raw.githubusercontent.com/SofyanBaharudinnn/heart-disease-prediction/main/dataset/heart.csv')

print(f'📊 Dataset berhasil dimuat: {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom')
print(f'\n{df_raw.head()}')

## 3️⃣ Data Cleaning & Preprocessing

In [ ]:
df = df_raw.copy()

# ── 1. Info Awal
print('='*60)
print('INFO DATASET AWAL')
print('='*60)
print(df.info())

# ── 2. Statistik Deskriptif
print('\n', '='*60)
print('STATISTIK DESKRIPTIF')
print('='*60)
print(df.describe())

# ── 3. Cek Missing Values
print('\n', '='*60)
print('MISSING VALUES PER KOLOM')
print('='*60)
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing': missing, 'Persen (%)': missing_pct})
print(missing_df[missing_df['Missing'] > 0] if missing_df['Missing'].sum() > 0 else 'Tidak ada missing values ✅')

# ── 4. Cek Duplikat
duplicates = df.duplicated().sum()
print(f'\nJumlah Duplikat: {duplicates}')

# ── 5. Pembersihan
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)

# ── 6. Perbaiki Target (inversi jika perlu)
# Dataset asli: 0=sakit, 1=tidak sakit → kita balik agar 1=sakit (standar medis)
if 'target' in df.columns:
    unique_target = df['target'].unique()
    print(f'\nNilai unik target sebelum koreksi: {sorted(unique_target)}')
    # Periksa distribusi — jika mayoritas 0 adalah sakit, balik
    # Uncomment baris berikut jika perlu inversi:
    # df['target'] = 1 - df['target']
    print(f'Distribusi target:\n{df["target"].value_counts()}')

# ── 7. Validasi Range Nilai
print('\n', '='*60)
print('VALIDASI RANGE NILAI')
print('='*60)
expected_ranges = {
    'age': (20, 90), 'sex': (0, 1), 'cp': (0, 3),
    'trestbps': (80, 220), 'chol': (100, 650), 'fbs': (0, 1),
    'restecg': (0, 2), 'thalach': (50, 250), 'exang': (0, 1),
    'oldpeak': (0, 10), 'slope': (0, 2), 'ca': (0, 4), 'thal': (0, 3)
}
for col, (lo, hi) in expected_ranges.items():
    if col in df.columns:
        outliers = ((df[col] < lo) | (df[col] > hi)).sum()
        status = '✅' if outliers == 0 else f'⚠️ {outliers} outlier'
        print(f'  {col:12s} [{lo:3d} - {hi:3d}]: {status}')

print(f'\n✅ Dataset bersih: {df.shape[0]} baris, {df.shape[1]} kolom')

## 4️⃣ Exploratory Data Analysis (EDA)

In [ ]:
# =============================================================
# EDA 1: Distribusi Target
# =============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Distribusi Kelas Target', fontsize=15, fontweight='bold', y=1.01)

target_counts = df['target'].value_counts().sort_index()
labels = ['Tidak Sakit (0)', 'Sakit Jantung (1)']

# Pie chart
axes[0].pie(
    target_counts, labels=labels, colors=PALETTE,
    autopct='%1.1f%%', startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 11}
)
axes[0].set_title('Proporsi Kelas', fontweight='bold')

# Bar chart
bars = axes[1].bar(labels, target_counts, color=PALETTE, edgecolor='white', linewidth=1.5, width=0.5)
for bar, count in zip(bars, target_counts):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
                 f'{count}\n({count/len(df)*100:.1f}%)',
                 ha='center', va='bottom', fontweight='bold', fontsize=11)
axes[1].set_ylabel('Jumlah Pasien', fontsize=11)
axes[1].set_title('Jumlah Per Kelas', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('eda_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nDistribusi target:')
print(target_counts.to_string())

In [ ]:
# =============================================================
# EDA 2: Distribusi Fitur Numerik
# =============================================================
numerical_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Distribusi Fitur Numerik berdasarkan Target', fontsize=14, fontweight='bold')

for i, col in enumerate(numerical_cols):
    # Histogram
    ax_hist = axes[0, i]
    for target_val, color, label in [(0, COLOR_NEG, 'Tidak Sakit'), (1, COLOR_POS, 'Sakit Jantung')]:
        subset = df[df['target'] == target_val][col]
        ax_hist.hist(subset, bins=20, alpha=0.6, color=color, label=label, edgecolor='white')
    ax_hist.set_title(col, fontweight='bold')
    ax_hist.set_xlabel(col)
    ax_hist.set_ylabel('Frekuensi')
    ax_hist.legend(fontsize=8)
    ax_hist.grid(alpha=0.3)

    # Boxplot
    ax_box = axes[1, i]
    data_to_plot = [df[df['target']==0][col].values, df[df['target']==1][col].values]
    bp = ax_box.boxplot(data_to_plot, labels=['Tidak Sakit', 'Sakit Jantung'],
                        patch_artist=True, notch=False,
                        medianprops={'color': 'white', 'linewidth': 2})
    for patch, color in zip(bp['boxes'], PALETTE):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax_box.set_title(f'{col} (Boxplot)', fontweight='bold')
    ax_box.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('eda_numerical_features.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================
# EDA 3: Fitur Kategorikal
# =============================================================
categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
col_labels = {
    'sex':     {0: 'Perempuan', 1: 'Laki-laki'},
    'cp':      {0: 'Typical', 1: 'Atypical', 2: 'Non-anginal', 3: 'Asymp'},
    'fbs':     {0: 'FBS ≤ 120', 1: 'FBS > 120'},
    'restecg': {0: 'Normal', 1: 'ST Abnormal', 2: 'LV Hypertrophy'},
    'exang':   {0: 'Tidak', 1: 'Ya'},
    'slope':   {0: 'Upsloping', 1: 'Flat', 2: 'Downsloping'},
    'ca':      {0: '0 vessel', 1: '1 vessel', 2: '2 vessels', 3: '3 vessels', 4: '4 vessels'},
    'thal':    {0: 'Normal', 1: 'Fixed Defect', 2: 'Reversible', 3: 'Unknown'},
}

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Distribusi Fitur Kategorikal berdasarkan Target', fontsize=14, fontweight='bold')

for i, col in enumerate(categorical_cols):
    ax = axes[i // 4, i % 4]
    ct = df.groupby([col, 'target']).size().unstack(fill_value=0)
    ct.plot(kind='bar', ax=ax, color=PALETTE, edgecolor='white', width=0.7)

    # Rename x-tick labels jika ada mapping
    if col in col_labels:
        ax.set_xticklabels(
            [col_labels[col].get(int(x.get_text()), x.get_text()) for x in ax.get_xticklabels()],
            rotation=25, ha='right', fontsize=8
        )
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Jumlah')
    ax.legend(['Tidak Sakit', 'Sakit Jantung'], fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('eda_categorical_features.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================
# EDA 4: Correlation Heatmap
# =============================================================
fig, ax = plt.subplots(figsize=(12, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # Sembunyikan segitiga atas

sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlBu_r', ax=ax,
    linewidths=0.5, linecolor='white',
    annot_kws={'size': 9},
    vmin=-1, vmax=1,
    square=True
)
ax.set_title('Korelasi Antar Fitur', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('eda_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Tampilkan fitur dengan korelasi tertinggi terhadap target
print('\nKorelasi terhadap TARGET (diurutkan):')
corr_target = corr['target'].drop('target').abs().sort_values(ascending=False)
for feat, val in corr_target.items():
    bar = '█' * int(val * 20)
    print(f'  {feat:12s}: {val:.4f} |{bar}')

## 5️⃣ Feature Encoding, Scaling & Train-Test Split

In [ ]:
# =============================================================
# 5A. Pisahkan Fitur & Target
# =============================================================
FEATURE_COLS = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
                'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
TARGET_COL   = 'target'

X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].copy()

print(f'Fitur (X): {X.shape}  |  Target (y): {y.shape}')
print(f'Fitur: {FEATURE_COLS}')
print(f'\nDistribusi kelas:')
print(y.value_counts().to_string())

# =============================================================
# 5B. Encoding (semua kolom sudah numerik, tidak perlu OHE)
# Cukup pastikan tipe data integer/float
# =============================================================
for col in FEATURE_COLS:
    X[col] = pd.to_numeric(X[col], errors='coerce').fillna(X[col].median())

print(f'\nTipe data setelah encoding:')
print(X.dtypes.to_string())

# =============================================================
# 5C. Train-Test Split (80:20, stratified)
# =============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTrain: {X_train.shape[0]} sampel | Test: {X_test.shape[0]} sampel')
print(f'Distribusi kelas TRAIN: {dict(y_train.value_counts().sort_index())}')
print(f'Distribusi kelas TEST : {dict(y_test.value_counts().sort_index())}')

# =============================================================
# 5D. Feature Scaling (StandardScaler)
# =============================================================
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # Fit HANYA pada data train
X_test_sc  = scaler.transform(X_test)        # Transform test pakai scaler yg sama

print(f'\n✅ Scaling selesai!')
print(f'Mean train (age): {scaler.mean_[0]:.4f} | Std: {scaler.scale_[0]:.4f}')

## 6️⃣ Training Random Forest (Baseline)
> Latih model dasar dulu sebelum hyperparameter tuning.

In [ ]:
# =============================================================
# 6A. Random Forest Baseline
# =============================================================
rf_baseline = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf_baseline.fit(X_train_sc, y_train)

y_pred_base  = rf_baseline.predict(X_test_sc)
y_prob_base  = rf_baseline.predict_proba(X_test_sc)[:, 1]

def evaluate_model(y_true, y_pred, y_prob, model_name='Model'):
    """Hitung dan tampilkan semua metrik evaluasi."""
    metrics = {
        'Accuracy':  accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall':    recall_score(y_true, y_pred, zero_division=0),
        'F1-Score':  f1_score(y_true, y_pred, zero_division=0),
        'ROC-AUC':   roc_auc_score(y_true, y_prob),
    }
    print(f'\n{"+":=<50}')
    print(f'  {model_name}')
    print(f'{"+":=<50}')
    for name, val in metrics.items():
        bar = '█' * int(val * 30)
        print(f'  {name:12s}: {val:.4f} |{bar}')
    print(f'\n  Classification Report:')
    print(classification_report(y_true, y_pred, target_names=['Tidak Sakit', 'Sakit Jantung']))
    return metrics

baseline_metrics = evaluate_model(y_test, y_pred_base, y_prob_base, 'Random Forest Baseline')

# =============================================================
# 6B. Cross-Validation (5-Fold Stratified)
# =============================================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf_baseline, X_train_sc, y_train, cv=skf, scoring='f1', n_jobs=-1)
print(f'\nCross-Validation F1 (5-Fold):')
for i, score in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {score:.4f}')
print(f'  Mean   : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## 7️⃣ Hyperparameter Tuning (GridSearchCV)
> Cari kombinasi hyperparameter terbaik untuk Random Forest.

In [ ]:
# =============================================================
# 7. GridSearchCV — Hyperparameter Tuning
# =============================================================
param_grid = {
    'n_estimators':      [50, 100, 200, 300],
    'max_depth':         [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2'],
    'class_weight':      [None, 'balanced'],
}

print(f'Grid parameter:')
for k, v in param_grid.items():
    print(f'  {k}: {v}')
total = 1
for v in param_grid.values():
    total *= len(v)
print(f'\nTotal kombinasi: {total} x 5 CV = {total*5} fits')
print('\n⏳ Menjalankan GridSearchCV... (mungkin 5-15 menit)')

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1',            # Optimasi berdasarkan F1-Score
    n_jobs=-1,
    verbose=1,
    refit=True               # Refit model terbaik ke seluruh data train
)

grid_search.fit(X_train_sc, y_train)

print(f'\n✅ GridSearchCV selesai!')
print(f'Best Score (F1 CV): {grid_search.best_score_:.4f}')
print(f'\nBest Parameters:')
for k, v in grid_search.best_params_.items():
    print(f'  {k}: {v}')

## 8️⃣ Evaluasi Model Terbaik
> Bandingkan Baseline vs Model Setelah Tuning.

In [ ]:
# =============================================================
# 8A. Evaluasi Model Setelah Tuning
# =============================================================
best_model = grid_search.best_estimator_

y_pred_tuned = best_model.predict(X_test_sc)
y_prob_tuned = best_model.predict_proba(X_test_sc)[:, 1]

tuned_metrics = evaluate_model(y_test, y_pred_tuned, y_prob_tuned, 'Random Forest (After Tuning)')

# =============================================================
# 8B. Tabel Perbandingan
# =============================================================
comparison = pd.DataFrame({
    'Metrik':   ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'Baseline': [baseline_metrics[m] for m in ['Accuracy','Precision','Recall','F1-Score','ROC-AUC']],
    'Tuned':    [tuned_metrics[m]    for m in ['Accuracy','Precision','Recall','F1-Score','ROC-AUC']],
})
comparison['Improvement'] = comparison['Tuned'] - comparison['Baseline']
comparison = comparison.set_index('Metrik')
comparison_fmt = comparison.applymap(lambda x: f'{x:.4f}' if isinstance(x, float) else x)
print('\nPerbandingan Baseline vs Tuned:')
print(comparison_fmt.to_string())

# Pilih model terbaik (tuned jika F1 lebih baik, baseline jika sama/lebih rendah)
if tuned_metrics['F1-Score'] >= baseline_metrics['F1-Score']:
    FINAL_MODEL = best_model
    print('\n✅ Model TUNED dipilih sebagai model final!')
else:
    FINAL_MODEL = rf_baseline
    print('\n✅ Model BASELINE dipilih (tuning tidak meningkatkan performa)!')

## 9️⃣ Visualisasi Hasil Evaluasi
> Confusion Matrix, ROC Curve, Feature Importance, dan Perbandingan Metrik.

In [ ]:
# =============================================================
# 9A. Confusion Matrix & ROC Curve (Side by Side)
# =============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred_tuned)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Tidak Sakit (0)', 'Sakit Jantung (1)'],
            yticklabels=['Tidak Sakit (0)', 'Sakit Jantung (1)'],
            ax=axes[0], linewidths=1.5, linecolor='white',
            annot_kws={'size': 12, 'weight': 'bold'})
axes[0].set_title('Confusion Matrix (Model Tuned)', fontsize=13, fontweight='bold', pad=10)
axes[0].set_ylabel('Aktual', fontsize=11)
axes[0].set_xlabel('Prediksi', fontsize=11)

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_tuned)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#e63946', lw=3, label=f'ROC Curve (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1.5, label='Random Classifier')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#e63946')
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate', fontsize=11)
axes[1].set_title('ROC Curve (Model Tuned)', fontsize=13, fontweight='bold', pad=10)
axes[1].legend(loc='lower right', fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation_plots.png', dpi=150, bbox_inches='tight')
plt.show()

# =============================================================
# 9B. Feature Importance Plot
# =============================================================
importances = pd.Series(FINAL_MODEL.feature_importances_, index=FEATURE_COLS)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
# Gunakan warna sekunder untuk fitur terpenting, aksen untuk yang lain
colors_bar = ['#e63946' if val >= importances.quantile(0.7) else '#2a9d8f' for val in importances.values]

ax.barh(importances.index, importances.values, color=colors_bar, edgecolor='white', height=0.6)
ax.set_title('Feature Importance – Random Forest Final', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Importance Score', fontsize=11)
ax.grid(axis='x', alpha=0.3)

# Tulis label nilai importance di ujung bar
for i, v in enumerate(importances.values):
    ax.text(v + 0.005, i, f'{v:.4f}', va='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nNilai Feature Importance (diurutkan):')
for feat, val in importances.sort_values(ascending=False).items():
    bar = '█' * int(val * 40)
    print(f'  {feat:12s}: {val:.4f} |{bar}')

## 🔟 Latih Model pada 100% Data & Simpan Model `.pkl`
> Untuk deployment production di website Django, kita harus melatih model menggunakan 100% data (tidak dibagi train-test split lagi) dengan hyperparameter optimal hasil tuning agar generalisasi model maksimal.
> Kita juga menyimpan `scaler` karena input pengguna saat prediksi harus di-scale menggunakan parameter scaling yang sama.

In [ ]:
from sklearn.base import clone
import joblib

print('='*60)
print('TRAINING MODEL FINAL PADA 100% DATA')
print('='*60)

# 1. Standarisasi seluruh dataset
final_scaler = StandardScaler()
X_scaled = final_scaler.fit_transform(X)

# 2. Clone model terpilih agar parameternya sama
final_model = clone(FINAL_MODEL)

# Latih pada seluruh data
final_model.fit(X_scaled, y)
print(f'Model final berhasil dilatih pada {X_scaled.shape[0]} sampel!')

# 3. Simpan ke .pkl
model_filename = 'random_forest_model.pkl'
scaler_filename = 'scaler.pkl'

joblib.dump(final_model, model_filename)
joblib.dump(final_scaler, scaler_filename)

print(f'\n✅ File berhasil disimpan:')
print(f'  - Model : {model_filename} ({os.path.getsize(model_filename) / 1024:.2f} KB)')
print(f'  - Scaler: {scaler_filename} ({os.path.getsize(scaler_filename) / 1024:.2f} KB)')

# 4. Download otomatis di Colab
try:
    from google.colab import files
    print('\nMendownload file secara otomatis...')
    files.download(model_filename)
    files.download(scaler_filename)
except ImportError:
    print('\n⚠️ Bukan di Google Colab. Silakan download file secara manual.')

## 1️⃣1️⃣ Uji Coba Prediksi dengan Model Tersimpan
> Memastikan model dan scaler yang disimpan berfungsi dengan baik untuk melakukan prediksi secara real-time. Kita akan membuat simulasi data pasien baru dan melakukan prediksi.

In [ ]:
# 1. Load model & scaler
loaded_model = joblib.load('random_forest_model.pkl')
loaded_scaler = joblib.load('scaler.pkl')

# 2. Definisikan sampel data pasien baru (13 fitur)
# Sesuai urutan FEATURE_COLS:
# ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
sample_patient = {
    'age': 57.0,
    'sex': 1.0,      # Laki-laki
    'cp': 0.0,       # Typical Angina
    'trestbps': 140.0,
    'chol': 192.0,
    'fbs': 0.0,
    'restecg': 1.0,
    'thalach': 148.0,
    'exang': 0.0,
    'oldpeak': 0.4,
    'slope': 1.0,
    'ca': 0.0,
    'thal': 1.0
}

# Konversi dict ke numpy array 2D
X_new = np.array([[sample_patient[col] for col in FEATURE_COLS]])

# 3. Transform menggunakan scaler yang di-load
X_new_sc = loaded_scaler.transform(X_new)

# 4. Prediksi
prediction = loaded_model.predict(X_new_sc)[0]
probabilities = loaded_model.predict_proba(X_new_sc)[0]

print('='*60)
print('SIMULASI DETEKSI PENYAKIT JANTUNG REAL-TIME')
print('='*60)
print(f'Data Pasien Baru: {sample_patient}')
print(f'Fitur ter-scale  : {X_new_sc}')
print('-'*60)
print(f'Hasil Prediksi   : {prediction} ({"Positif Penyakit Jantung" if prediction == 1 else "Negatif Penyakit Jantung"})')
print(f'Probabilitas     :')
print(f'  - Negatif: {probabilities[0]*100:.2f}%')
print(f'  - Positif: {probabilities[1]*100:.2f}%')
print('='*60)